# 03 — Pollinator Occurrences (Small-Scale Testing)

Loads `pollinator_observations_v2.csv` and builds a spatial density map
for the top 50 pollinator species by observation count.
Used for exploratory visualization only.

**Data source:** `pollinator_observations_v2.csv` (25,466 species, CONUS, 2013–2026)

**Note on observation bias:** GBIF pollinator records are opportunistic citizen
science observations. Observation density in any spatial bin reflects recorder
activity, not genuine pollinator abundance or phenological activity. Regions
near major cities and accessible areas are systematically overrepresented.
This bias is explicitly documented in the limitations and motivates the SDM
integration (replacing GBIF-derived activity curves with model-predicted ones).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

BASE       = Path("/scratch/ariana.l")
POLL_OBS   = BASE / "Plant Pollinator Initial Analysis" / "pollinator_observations_v2.csv"
OUT_DIR    = BASE / "Plant Pollinator Initial Analysis"

BIN_SIZE   = 0.5
TOP_N      = 50

print("Paths OK")

In [ ]:
# Load pollinator observations
print("Loading pollinator observations...")
pollinators = pd.read_csv(POLL_OBS, low_memory=False)
print(f"  Records: {len(pollinators):,}")
print(f"  Species: {pollinators['pollinator_species'].nunique():,}")
pollinators.head()

In [ ]:
# Select top 50 species by observation count
top50 = pollinators['pollinator_species'].value_counts().head(TOP_N).index
pol_top50 = pollinators[pollinators['pollinator_species'].isin(top50)].copy()

print(f"Top {TOP_N} species: {len(pol_top50):,} records")
print(pollinators['pollinator_species'].value_counts().head(10))

In [ ]:
# Add spatial bins
pol_top50['lat_bin'] = (np.floor(pol_top50['lat'] / BIN_SIZE) * BIN_SIZE).round(1)
pol_top50['lon_bin'] = (np.floor(pol_top50['lon'] / BIN_SIZE) * BIN_SIZE).round(1)
pol_top50['bin']     = pol_top50['lat_bin'].astype(str) + '_' + pol_top50['lon_bin'].astype(str)

# Add week
pol_top50['week'] = ((pol_top50['doy'].astype(int) - 1) // 7).clip(0, 51)

bin_counts = pol_top50.groupby(['lat_bin', 'lon_bin']).size().reset_index(name='count')
print(f"  Unique bins: {len(bin_counts):,}")

In [ ]:
# Plot observation density map
# Note: density reflects observer effort, not true pollinator distribution
fig, ax = plt.subplots(figsize=(14, 8))
scatter = ax.scatter(
    bin_counts['lon_bin'], bin_counts['lat_bin'],
    c=bin_counts['count'], cmap='YlOrRd',
    s=50, alpha=0.8
)
plt.colorbar(scatter, ax=ax, label='Observation count (proxy for recorder effort)')
ax.set_xlim(-130, -60)
ax.set_ylim(24, 50)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(
    f'Top {TOP_N} Pollinator Species — Observations per {BIN_SIZE}°×{BIN_SIZE}° bin\n'
    '(Density reflects iNaturalist recorder effort, not true pollinator abundance)'
)
plt.tight_layout()
plt.savefig(OUT_DIR / 'pollinator_density_top50.png', dpi=150)
print("Saved.")